# Butly LoCoMo Evaluation (Colab Pro)

This notebook is a **thin frontend**: it mounts Drive, prepares the repo and
local model servers, then drives `python -m evals.locomo.cli`. All evaluation
logic lives in `evals/locomo/` — do not add scoring, replay, or checkpoint
code here.

**Role-based model servers.** Each Butly role (chat / gatekeeper / summary /
knowledge / embedding) can use its own model, configured in the Parameters
cell. Roles that share the same model + port share one server. Reasoning
(thinking) models are accurate but slow; assign Non-Reasoning models to
gatekeeper / summary / knowledge for practical throughput. Embeddings need a
dedicated embeddings-capable server (a chat LLM cannot answer
`/v1/embeddings`).

Prerequisites:

* Use a **GPU runtime** (Runtime -> Change runtime type -> GPU; L4 is fine).
* Put the LoCoMo dataset JSON on Drive. Official data is CC BY-NC 4.0 and is
  **not** bundled with Butly — download it from
  https://github.com/snap-research/locomo yourself.
* Optionally add `HF_TOKEN` to Colab Secrets for gated/rate-limited downloads.

llama.cpp is built fresh each session (a few minutes); Drive binary caching
was removed after repeated shared-library breakage. Artifacts (checkpoints
included) are written to Drive, so a disconnected runtime can continue with
the **Resume** cell near the end.

The Parameters form defaults to independent QA and English. Set all three
`ALL_*` switches to run every LoCoMo sample, session, and question; otherwise
the corresponding positive limit is passed to the CLI.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Butly LoCoMo parameters
# --- Parameters (edit these) ---
REPO_URL = 'https://github.com/unagisann/Butly.git'
BRANCH = 'main'
REPO_DIR = '/content/butly'

DRIVE_ROOT = '/content/drive/MyDrive/butly-evals'
DATASET_PATH = f'{DRIVE_ROOT}/data/locomo10.json'
RUN_ID = 'qwen3_14b_colab_run_01'   # one run directory per model + attempt

# --- Evaluation policy and scope ---
# independent: every QA starts from the same post-Sleeptime memory state.
# sequential: QA turns accumulate, for operational/endurance evaluation.
QA_MODE = 'independent'  # @param ["independent", "sequential"]
# Internal prompt/memory language; LoCoMo questions and gold stay unchanged.
# QA answers stay English for compatibility with the official scorer.
LOCALE = 'en'  # @param ["en", "ja"]

# An ALL_* switch overrides its corresponding positive LIMIT value.
# Full LoCoMo = set all three switches to True.
ALL_SAMPLES = False  # @param {type:"boolean"}
SAMPLE_LIMIT = 1  # @param {type:"integer", min:1}
ALL_SESSIONS = False  # @param {type:"boolean"}
SESSION_LIMIT = 3  # @param {type:"integer", min:1}
ALL_QUESTIONS = False  # @param {type:"boolean"}
QUESTION_LIMIT = 10  # @param {type:"integer", min:1}

# --- Model roles ---
# Roles sharing the same hf_repo/hf_file/port share one server process.
# chat is the model under evaluation. gatekeeper/summary/knowledge default to
# the same Qwen3 server (v8/v9 showed a separate Non-Reasoning Gemma4 pipeline
# scored worse and was not faster); swap hf_repo/hf_file/port per role to try
# other models. VRAM guide for L4 (22.5GB): Qwen3-14B Q4 ~9GB + KV ~1.3GB
# + embedding ~0.3GB; adding a 7B Q4 server costs ~6GB more and still fits.
# Optional per-role keys:
#   ctx               server context length (default 8192). KV cache VRAM
#                     grows with ctx and can OOM the later servers.
#   generation_config per-role overrides written into the profile. Reasoning
#                     models think before answering, and thinking alone can
#                     exhaust the default 512-token output budget -> empty
#                     classifications (v9: 8/10 questions). 2048 gives the
#                     classifier room; the JSON itself needs only ~100 tokens.
MODELS = {
    'chat': dict(
        hf_repo='Qwen/Qwen3-14B-GGUF',
        hf_file='Qwen3-14B-Q4_K_M.gguf',
        model_name='qwen3-14b',
        port=8090,
    ),
    'gatekeeper': dict(
        hf_repo='Qwen/Qwen3-14B-GGUF',
        hf_file='Qwen3-14B-Q4_K_M.gguf',
        model_name='qwen3-14b',
        port=8090,
        generation_config=dict(max_output_tokens=2048),
    ),
    'summary': dict(
        hf_repo='Qwen/Qwen3-14B-GGUF',
        hf_file='Qwen3-14B-Q4_K_M.gguf',
        model_name='qwen3-14b',
        port=8090,
    ),
    'knowledge': dict(
        hf_repo='Qwen/Qwen3-14B-GGUF',
        hf_file='Qwen3-14B-Q4_K_M.gguf',
        model_name='qwen3-14b',
        port=8090,
    ),
    'embedding': dict(
        hf_repo='nomic-ai/nomic-embed-text-v1.5-GGUF',
        hf_file='nomic-embed-text-v1.5.Q4_K_M.gguf',
        model_name='nomic-embed-text',
        port=8091,
        embeddings=True,
    ),
}

# Non-model instance-config overrides written into the profile as-is.
# v12: inject the RAG cards AND the original conversation excerpts they were
# built from (parent-document retrieval via each card's source_files) --
# cards stay lossy, the excerpts carry the exact dates and status facts.
# rag_source_mode: 'cards' (pre-v12) | 'raw' | 'both'. Excerpts are capped
# at rag_raw_max_chars characters in total (0 = unlimited).
PROFILE_EXTRAS = {
    'memory': dict(
        rag_source_mode='both',
        rag_raw_max_chars=6000,
    ),
}

# 1 server per port; reject conflicting assignments
server_specs = {}
for role, spec in MODELS.items():
    existing = server_specs.get(spec['port'])
    if existing and (existing['hf_repo'], existing['hf_file']) != (spec['hf_repo'], spec['hf_file']):
        raise ValueError(f"port {spec['port']} is assigned two different models")
    merged = dict(existing or {})
    merged.update(spec)
    server_specs[spec['port']] = merged
print('servers:', {port: s['model_name'] for port, s in sorted(server_specs.items())})


In [ ]:
# --- Clone / update Butly and install dependencies ---
import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
# --- Build llama.cpp (fresh each session, run in place from build/bin) ---
# No Drive caching: cached binaries repeatedly broke on missing shared
# libraries. Running from build/bin keeps the RPATH valid.
import os
!apt-get -qq install -y libcurl4-openssl-dev > /dev/null
![ -d /content/llama.cpp ] || git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-server -j > /dev/null
SERVER_BIN = '/content/llama.cpp/build/bin/llama-server'
print('binary exists:', os.path.isfile(SERVER_BIN))


In [ ]:
# --- Download the GGUF models for every server ---
import os
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass  # token only needed for gated / rate-limited downloads

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
for port, spec in sorted(server_specs.items()):
    spec['model_path'] = hf_hub_download(spec['hf_repo'], spec['hf_file'])
    print(port, spec['model_name'], '->', spec['model_path'])


In [ ]:
# --- Robust server launcher: kill stale server on the port, log to file,
#     wait for /health, surface the log on failure ---
import subprocess, time, socket, pathlib, urllib.request

def _port_free(port):
    s = socket.socket()
    # TIME_WAIT ソケット(直前にkillしたサーバーの残骸)を空きとみなす
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        s.bind(('127.0.0.1', port)); return True
    except OSError:
        return False
    finally:
        s.close()

def start_llama_server(model_path, port, extra_args=None, timeout=600):
    extra_args = extra_args or []
    # free the port (kill a previous llama-server bound to it)
    # パターンを '--' 始まりにしない: pkill が自身のオプションと誤解釈して
    # 何も殺さずエラー終了する
    subprocess.run(
        ['pkill', '-9', '-f', f'llama-server.*--port {port}'],
        capture_output=True,
    )
    for _ in range(15):
        if _port_free(port):
            break
        time.sleep(2)
    else:
        raise RuntimeError(f'port {port} is busy and did not free up; pick another in Parameters')

    log_path = f'/content/llama_{port}.log'
    log = open(log_path, 'w')
    proc = subprocess.Popen(
        [SERVER_BIN, '-m', model_path, '--port', str(port), '-ngl', '99'] + extra_args,
        stdout=log, stderr=subprocess.STDOUT,
    )
    for i in range(timeout // 2):
        if proc.poll() is not None:
            print(f'server on {port} EXITED with code', proc.returncode)
            print(pathlib.Path(log_path).read_text()[-3000:])
            raise RuntimeError(f'llama-server on {port} exited; see log above')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=2)
            print(f'server on {port} is UP')
            return proc
        except Exception:
            if i % 15 == 14:
                tail = pathlib.Path(log_path).read_text().splitlines()
                print(f'  {port} loading...', tail[-1] if tail else '(no output yet)')
            time.sleep(2)
    raise RuntimeError(f'server on {port} did not become healthy in {timeout}s (see /content/llama_{port}.log)')

server_procs = {}
for port, spec in sorted(server_specs.items()):
    extra = ['--embeddings', '--pooling', 'mean'] if spec.get('embeddings') else []
    # コンテキスト長は明示する。未指定だとモデル既定(Qwen3は40K等)でKVキャッシュを
    # 確保してVRAMを浪費し、後続サーバーがOOMで起動失敗する。
    extra += ['-c', str(spec.get('ctx', 8192))]
    server_procs[port] = start_llama_server(spec['model_path'], port, extra_args=extra)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
# --- Register each server as a Butly connection + write the role profile ---
import json, os, pathlib, urllib.request
import yaml

user_config = {
    'LLM_CONNECTIONS': [
        {
            'id': f'colab_{port}',
            'protocol': 'openai_compat',
            'base_url': f'http://127.0.0.1:{port}/v1',
            'api_key_env': 'COLAB_LOCAL_API_KEY',
            'label': f"Colab {spec['model_name']} (:{port})",
        }
        for port, spec in sorted(server_specs.items())
    ]
}
pathlib.Path('user_config.json').write_text(json.dumps(user_config, indent=2))
os.environ['COLAB_LOCAL_API_KEY'] = 'local'  # llama.cpp accepts any key

profile = {'name': 'colab_roles', 'locale': LOCALE}
for role, spec in MODELS.items():
    entry = {
        'connection': f"colab_{spec['port']}",
        'model_name': spec['model_name'],
    }
    # per-role generation overrides (see MODELS comment in the Parameters cell)
    if spec.get('generation_config'):
        entry['generation_config'] = dict(spec['generation_config'])
    profile[role] = entry
# non-model sections (memory etc. -- see PROFILE_EXTRAS in the Parameters cell)
for section, overrides in PROFILE_EXTRAS.items():
    profile[section] = dict(overrides)
pathlib.Path('evals/locomo/profiles/colab_roles.yaml').write_text(
    yaml.safe_dump(profile, sort_keys=False)
)
print(yaml.safe_dump(profile, sort_keys=False))

# sanity check: every server answers /health
for port in sorted(server_specs):
    body = urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=3).read().decode()
    print(f':{port} /health:', body)
print('connections + profile ready')


In [ ]:
# --- Run the evaluation (replay -> sleeptime -> QA -> score -> report) ---
import shlex, subprocess, sys

scope_args = []
for dimension, use_all, limit in (
    ('samples', ALL_SAMPLES, SAMPLE_LIMIT),
    ('sessions', ALL_SESSIONS, SESSION_LIMIT),
    ('questions', ALL_QUESTIONS, QUESTION_LIMIT),
):
    if use_all:
        scope_args.append(f'--all-{dimension}')
    else:
        scope_args.extend([f'--{dimension[:-1]}-limit', str(limit)])

command = [
    sys.executable, '-m', 'evals.locomo.cli', 'run',
    '--dataset', DATASET_PATH,
    '--output-dir', f'{DRIVE_ROOT}/runs',
    '--run-id', RUN_ID,
    '--profile', 'evals/locomo/profiles/colab_roles.yaml',
    '--qa-mode', QA_MODE,
    '--locale', LOCALE,
    *scope_args,
]
print(shlex.join(command))
subprocess.run(command, check=True)


In [ ]:
# --- Resume after a runtime disconnect (safe to re-run; skips finished work) ---
# Re-run the setup cells above first (mount, clone, build, download, servers,
# connections), then run this cell instead of the run cell.
!python -m evals.locomo.cli resume --run-dir "{DRIVE_ROOT}/runs/{RUN_ID}"

In [ ]:
# --- Show the summary ---
# Guarded so a catastrophically failed run still reaches the teardown cell
# below (otherwise the raised exception stops "Run all" and the GPU idles).
from IPython.display import Markdown, display
import pathlib
summary_path = pathlib.Path(f'{DRIVE_ROOT}/runs/{RUN_ID}/summary.md')
if summary_path.is_file():
    display(Markdown(summary_path.read_text()))
else:
    print(f'summary.md not found: {summary_path} (run failed early?)')

In [ ]:
# --- Disconnect and delete the runtime (stop consuming GPU / compute units) ---
# Run this last. It flushes pending Drive writes first so the run results are
# safely persisted, then unassigns the runtime. With "Run all", the session
# frees itself automatically once the evaluation and summary have finished --
# no idle GPU burning compute units overnight.
# Skip this cell if you want to keep the session for interactive inspection.
from google.colab import drive, runtime

try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted.')
except Exception as e:
    print(f'Drive flush skipped: {e}')

runtime.unassign()